# Chapter 04 — APIs, Web Data, and Schema Validation

External systems expose data through HTTP APIs or web pages. Learn HTTP request structure, JSON response handling, defensive parsing, HTML extraction, and schema validation.

## Prerequisites
Dictionaries, JSON, exceptions, and basic DataFrames; live internet access is optional.

## Learning goals
- Explain URLs, methods, status codes, headers, query parameters, and timeouts.
- Use Requests to retrieve and check API responses.
- Parse HTML locally with BeautifulSoup.
- Validate records with Pydantic and report malformed data.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Practical scenario

Build a local ingestion pipeline that reads an API-shaped JSON fixture, parses an HTML catalogue, validates products, and creates a clean DataFrame.

## Tasks to complete

- Check the JSON response status and record count.
- Extract products with a CSS selector from the provided HTML.
- Define positive-price and non-empty-name validation.
- Separate valid and invalid records.
- Explain how a live API function should handle timeout and HTTP errors.

**Working rule:** write your own code in the next cell before opening the reference solution. Explain each assumption and validate the result.

In [2]:
# TODO: complete the practical lab here.
# Start by loading the relevant dataset or defining the required input.
# Add validation checks before calculating the requested outputs.

## Worked reference solution

The following is one valid approach. It is not the only possible solution. Compare your work with it, and explain any differences in assumptions, handling, or output.

In [3]:
import json, pandas as pd
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field, ValidationError
payload=json.loads((DATA/"sample_api_response.json").read_text(encoding="utf-8"))
class Product(BaseModel):
 id:int
 name:str=Field(min_length=1)
 price:float=Field(gt=0)
valid=[]; invalid=[]
for record in payload.get("results",[]):
 try:
  item=Product.parse_obj(record); valid.append(item.dict() if hasattr(item,"dict") else item.model_dump())
 except ValidationError as e: invalid.append({"record":record,"error":str(e)})
soup=BeautifulSoup((DATA/"sample_page.html").read_text(encoding="utf-8"),"html.parser")
html_items=[{"id":int(x["data-id"]),"label":x.get_text(" ",strip=True)} for x in soup.select("li.product[data-id]")]
print("Valid:",len(valid),"invalid:",len(invalid),"HTML items:",len(html_items)); display(pd.DataFrame(valid))

Valid: 3 invalid: 0 HTML items: 3
   id      name  price
0   1  Notebook  120.0
1   2       Pen   15.0
2   3  Backpack  850.0


/mnt/data/python_data_analysis_book/Chapter_04_apis_web_data_and_schema_validation/Chapter_04_Practical_Lab_and_Exercises.ipynb:12: PydanticDeprecatedSince20:

The `parse_obj` method is deprecated; use `model_validate` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/

/mnt/data/python_data_analysis_book/Chapter_04_apis_web_data_and_schema_validation/Chapter_04_Practical_Lab_and_Exercises.ipynb:12: PydanticDeprecatedSince20:

The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/



## Extension challenge

Add a paginated API workflow using `next_page` tokens. Respect retry-after headers and do not retry schema-validation failures as network errors.

## Further reading

- Requests: https://requests.readthedocs.io/
- BeautifulSoup: https://www.crummy.com/software/BeautifulSoup/bs4/doc/
- Pydantic: https://docs.pydantic.dev/

## Completion checklist

- [ ] I can explain the topic without reading the cells.
- [ ] I can reproduce the examples from a clean kernel.
- [ ] I completed each task and checked the reference solution.
- [ ] I tested an edge case.
- [ ] I can describe at least one limitation or risk.